# 06 · Enriching the content-based recommender

Content-v1 uses genres and release decade. These features cover most
movies but cannot distinguish many movies with identical metadata.

This notebook investigates whether external MovieLens tags and TMDB
metadata can improve rare-movie recommendations.

We will first measure tag coverage, then retrieve and cache richer
movie metadata, build content features, and evaluate every candidate
against the frozen content-v1 baseline.

All feature choices are made using validation data. The test set is
reserved for the final evaluation.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "scripts"))

import numpy as np
import pandas as pd

from recutils import load_split

NAME = "sample_10pct"
REL = 4.0
SEED = 42

S = load_split(NAME)

train = S["train"]
val = S["val"]
test = S["test"]

n_users = int(S["user_map"]["u"].max()) + 1
n_items = int(S["item_map"]["i"].max()) + 1

movies = pd.read_parquet(
    Path("../data/interim") / NAME / "movies.parquet"
)

meta = (
    S["item_map"]
    .merge(movies, on="movieId", how="left")
    .set_index("i")
    .sort_index()
)

assert len(meta) == n_items
assert meta["title"].notna().all()

print(f"Users: {n_users:,}")
print(f"Movies: {n_items:,}")
print("Train:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

Users: 20,093
Movies: 17,241
Train: (2919680, 4)
Validation: (137197, 4)
Test: (137197, 4)


## 1 · How well are rare movies covered by external tags?

We group movies by the number of positive training ratings they received.
A positive rating is one rated 4 or higher.

We then measure how many movies in each group have external tags and
how many distinct tags they have. This tells us whether tag features
are likely to provide useful information for the rare movies we care about.

In [2]:
n_pos = np.bincount(
    train.i[train.rating >= REL],
    minlength=n_items,
)

bins = [-1, 0, 4, 19, 99, 499, np.inf]

labels = [
    "0",
    "1-4",
    "5-19",
    "20-99",
    "100-499",
    "500+",
]

bucket = pd.Series(
    pd.cut(n_pos, bins=bins, labels=labels),
    index=meta.index,
    name="bucket",
)

display(
    bucket.value_counts()
    .reindex(labels)
    .rename("movies")
    .to_frame()
)

,movies
bucket,
0,1095
1-4,5666
5-19,4836
20-99,3299
100-499,1685
500+,660


## 2 · Build a leak-safe external tag dataset

The sampled users belong to our model-training and evaluation population.
To avoid directly using their tagging behaviour as movie features, we
build the tag vocabulary only from users outside that sample.

We keep tags for the modelled catalogue and map them to the same internal
movie indices used by the recommender.

In [3]:
RAW = Path("../data/raw/ml-32m")
tag_path = RAW / "tags.csv"

assert tag_path.exists(), (
    f"Cannot find {tag_path}. Check your raw MovieLens folder."
)

print("User map columns:", S["user_map"].columns.tolist())

raw_tags = pd.read_csv(
    tag_path,
    dtype={
        "userId": "int32",
        "movieId": "int32",
    },
)

print("Raw tag rows:", f"{len(raw_tags):,}")
print("Tag columns:", raw_tags.columns.tolist())

User map columns: ['u', 'userId']
Raw tag rows: 2,000,072
Tag columns: ['userId', 'movieId', 'tag', 'timestamp']


### 2.1 · Exclude sampled users and map the tags

We remove tags created by sampled users, clean the tag text, and join
the remaining records to the modelled catalogue.

Tags belonging to movies outside the catalogue are excluded because
our current evaluation cannot score those movies.

In [4]:
assert "userId" in S["user_map"].columns

sampled_user_ids = set(
    S["user_map"]["userId"].astype(int)
)

external_tags = raw_tags.loc[
    ~raw_tags["userId"].isin(sampled_user_ids),
    ["userId", "movieId", "tag"],
].copy()

external_tags["tag"] = (
    external_tags["tag"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

external_tags = external_tags.loc[
    external_tags["tag"].ne("")
]

tag_map = external_tags.merge(
    S["item_map"][["movieId", "i"]],
    on="movieId",
    how="inner",
    validate="many_to_one",
)

assert not tag_map["userId"].isin(sampled_user_ids).any()

print("External tag rows:", f"{len(external_tags):,}")
print("Mapped tag rows:", f"{len(tag_map):,}")
print("Movies with external tags:", f"{tag_map['i'].nunique():,}")

External tag rows: 1,905,595
Mapped tag rows: 1,660,566
Movies with external tags: 16,725


### 2.2 · Measure tag coverage by rarity

For each movie, we count the number of distinct external tags.
We then summarize the number and percentage of tagged movies in
each rarity group.

Coverage alone does not prove that tags improve recommendations.
It tells us whether enough rare movies have tag information to justify
building and evaluating a tag-based feature set.

In [5]:
external_tag_counts = tag_map.groupby("i")["tag"].nunique()

tag_count_per_movie = (
    external_tag_counts
    .reindex(meta.index, fill_value=0)
    .astype(int)
)

coverage = pd.DataFrame({
    "bucket": pd.Categorical(
        bucket.to_numpy(),
        categories=labels,
        ordered=True,
    ),
    "distinct_external_tags": tag_count_per_movie.to_numpy(),
})

tag_coverage = coverage.groupby(
    "bucket",
    observed=False,
).agg(
    movies=("distinct_external_tags", "size"),
    movies_with_tags=(
        "distinct_external_tags",
        lambda s: int((s > 0).sum()),
    ),
    mean_tags_per_movie=("distinct_external_tags", "mean"),
    median_tags_per_movie=("distinct_external_tags", "median"),
)

tag_coverage["pct_movies_with_tags"] = (
    100 * tag_coverage["movies_with_tags"] / tag_coverage["movies"]
)

display(tag_coverage.round(2))

,movies,movies_with_tags,mean_tags_per_movie,median_tags_per_movie,pct_movies_with_tags
bucket,,,,,
0,1095,996,15.08,7.0,90.96
1-4,5666,5330,19.13,8.0,94.07
5-19,4836,4758,34.92,16.0,98.39
20-99,3299,3296,62.12,30.0,99.91
100-499,1685,1685,108.53,58.0,100.00
500+,660,660,214.98,145.0,100.00


### 2.3 · Check tag timestamps

Our ratings use a per-user temporal split. External tags come from
different users, but some may have been created after held-out ratings.

We inspect the tag timestamps before building features so we understand
this limitation and document the temporal assumptions of our evaluation.

In [6]:
# Rebuild the external-tag table while retaining timestamps.
external_tag_events = raw_tags.loc[
    ~raw_tags["userId"].isin(sampled_user_ids),
    ["userId", "movieId", "tag", "timestamp"],
].merge(
    S["item_map"][["movieId", "i"]],
    on="movieId",
    how="inner",
    validate="many_to_one",
)

# Compare tag timestamps with the rating split.
milestones = pd.DataFrame({
    "event": [
        "Latest training interaction",
        "Earliest validation interaction",
        "Median validation interaction",
        "Earliest test interaction",
    ],
    "timestamp": [
        train["timestamp"].max(),
        val["timestamp"].min(),
        val["timestamp"].median(),
        test["timestamp"].min(),
    ],
})

milestones["date"] = pd.to_datetime(
    milestones["timestamp"],
    unit="s",
)

display(milestones)

# These are global comparisons, not proof of per-user temporal safety.
tag_times = external_tag_events["timestamp"]

temporal_summary = pd.Series({
    "External tag records in catalogue": len(tag_times),
    "Tags at/before latest train timestamp": (
        tag_times <= train["timestamp"].max()
    ).mean() * 100,
    "Tags at/before earliest validation timestamp": (
        tag_times <= val["timestamp"].min()
    ).mean() * 100,
})

display(temporal_summary.round(2))

,event,timestamp,date
0,Latest training interaction,1.697070e+09,2023-10-12 00:22:06
1,Earliest validation interaction,8.256384e+08,1996-03-01 00:00:00
2,Median validation interaction,1.247003e+09,2009-07-07 21:50:38
3,Earliest test interaction,8.272242e+08,1996-03-19 08:29:47


External tag records in catalogue               1660573.00
Tags at/before latest train timestamp                99.99
Tags at/before earliest validation timestamp          0.00
dtype: float64

### 2.3 · Tag coverage and temporal limitations

External MovieLens tags cover 90.96% of movies with zero positive
training ratings and 98.39% of movies with 5–19 positive ratings.
This makes tags a promising source of additional content information.

However, the ratings use a per-user temporal split, so there is no
single global validation timestamp. External tags may have been
created after some held-out ratings, even though they come from users
outside our sampled population.

For the primary content-v2 experiment, we will first evaluate TMDB
descriptive metadata. External tags remain a potential additional
experiment, subject to an appropriate temporal policy.

We will not claim that user-disjoint tags are automatically
temporally leakage-free.

## TMDB has API restrictions so switching to title based TF-IDF approach

## 3 · Content-v2a: title-based TF-IDF

Content-v1 uses genres and release decade, but many movies share the
same feature vector. Movie titles may provide additional distinctions,
such as franchise names and distinctive phrases.

We will convert titles into TF-IDF vectors using individual words and
two-word phrases. We remove the trailing release year because decade
is already represented in content-v1, normalize titles such as
"Matrix, The", and retain other parenthetical text that may contain
an alternate title.

Terms appearing in only one movie are excluded because they provide
little useful cross-movie similarity.

Title features are a candidate addition, not an assumed improvement.
We will evaluate them against frozen content-v1 using validation only.

In [9]:
import re
import numpy as np
import pandas as pd

assert "title" in meta.columns
assert len(meta) == n_items

def clean_movie_title(title):
    if pd.isna(title):
        return ""

    text = str(title).strip()

    # Remove only a trailing four-digit release year.
    text = re.sub(r"\s*\(\d{4}\)\s*$", "", text)

    # Normalize MovieLens titles such as "Matrix, The".
    match = re.match(
        r"^(?P<name>.+),\s*(?P<article>The|A|An)$",
        text,
        flags=re.IGNORECASE,
    )

    if match:
        article = match.group("article").title()
        name = match.group("name").strip()
        text = f"{article} {name}"

    return text.strip()


meta["title_clean"] = meta["title"].map(clean_movie_title)

print("Original titles:")
display(meta[["title", "title_clean"]].head(10))

print("Titles with a trailing article:")

display(
    meta.loc[
        meta["title"].str.contains(
            r",\s*(?:The|A|An)\s*(?:\(\d{4}\))?$",
            case=False,
            regex=True,
            na=False,
        ),
        ["title", "title_clean"],
    ].head(10)
)

assert meta["title_clean"].notna().all()

Original titles:


,title,title_clean
i,,
0,Toy Story (1995),Toy Story
1,Jumanji (1995),Jumanji
2,Grumpier Old Men (1995),Grumpier Old Men
3,Waiting to Exhale (1995),Waiting to Exhale
4,Father of the Bride Part II (1995),Father of the Bride Part II
5,Heat (1995),Heat
6,Sabrina (1995),Sabrina
7,Tom and Huck (1995),Tom and Huck
8,Sudden Death (1995),Sudden Death


Titles with a trailing article:


,title,title_clean
i,,
10,"American President, The (1995)",The American President
49,"Usual Suspects, The (1995)",The Usual Suspects
53,"Big Green, The (1995)",The Big Green
59,"Indian in the Cupboard, The (1995)",The Indian in the Cupboard
77,"Crossing Guard, The (1995)",The Crossing Guard
78,"Juror, The (1996)",The Juror
103,"Bridges of Madison County, The (1995)",The Bridges of Madison County
113,"Young Poisoner's Handbook, The (1995)",The Young Poisoner's Handbook
117,"Boys of St. Vincent, The (1992)",The Boys of St. Vincent


### 3.1 · Convert cleaned titles into TF-IDF features

Each movie title is converted into a sparse numerical vector.

We use unigrams (single words) and bigrams (two-word phrases), and
require each term to occur in at least two distinct movie titles.
This avoids spending features on terms that cannot create similarity
between different titles.

The resulting matrix follows the catalogue's internal movie-index
order, so each feature row corresponds to the correct movie.

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

title_vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    norm="l2",
    sublinear_tf=True,
    token_pattern=r"(?u)\b\w+\b",
    dtype=np.float32,
)

T_title = title_vectorizer.fit_transform(
    meta["title_clean"]
)

print("Title matrix shape:", T_title.shape)
print("Vocabulary size:", len(title_vectorizer.vocabulary_))
print("Non-zero entries:", f"{T_title.nnz:,}")
print(
    "Matrix density:",
    f"{T_title.nnz / (T_title.shape[0] * T_title.shape[1]):.4%}",
)

assert T_title.shape[0] == n_items
assert T_title.shape[1] > 0
assert np.isfinite(T_title.data).all()

row_norms = np.sqrt(
    T_title.multiply(T_title).sum(axis=1)
).A1

has_features = row_norms > 0

print("Movies with title features:", int(has_features.sum()))
print("Movies with zero title features:", int((~has_features).sum()))

assert np.allclose(row_norms[has_features], 1.0, atol=1e-5)

Title matrix shape: (17241, 9367)
Vocabulary size: 9367
Non-zero entries: 64,716
Matrix density: 0.0401%
Movies with title features: 15391
Movies with zero title features: 1850


### 3.2 · Inspect the learned title vocabulary

Before combining title features with content-v1, we inspect the terms
the vectorizer retained and check that the matrix is aligned with the
movie catalogue.

This verifies that the title representation is usable; it does not
establish that it improves recommendation quality.

In [12]:
feature_names = np.asarray(title_vectorizer.get_feature_names_out())

print("Example vocabulary terms:")
print(feature_names[:50].tolist())

print("\nMost distinctive terms by IDF:")
idf_order = np.argsort(title_vectorizer.idf_)[::-1][:20]

display(pd.DataFrame({
    "term": feature_names[idf_order],
    "idf": title_vectorizer.idf_[idf_order],
}))

assert len(meta) == T_title.shape[0]
assert meta.index.is_unique
assert np.array_equal(meta.index.to_numpy(), np.arange(n_items))

Example vocabulary terms:
['0', '0 you', '000', '000 leagues', '01', '1', '1 0', '1 2', '10', '10 to', '100', '1000', '101', '101 dalmatians', '10th', '11', '117', '12', '12 angry', '12 chairs', '12 rounds', '120', '12th', '13', '13th', '13th part', '14', '15', '16', '17', '18', '19', '1900', '1978', '1984', '1985', '2', '2 0', '2 a', '2 back', '2 days', '2 les', '2 on', '2 red', '2 the', '20', '20 000', '20 million', '200', '2000']

Most distinctive terms by IDF:


,term,idf
0,zong,9.656491
1,0 you,9.656491
2,zombieland,9.656491
3,zu,9.656491
4,zombi,9.656491
5,zoolander,9.656491
6,zookeeper,9.656491
7,12 angry,9.656491
8,12 chairs,9.656491
9,12 rounds,9.656491


### 3.3 · Which movies have usable title features?

Some movie titles contain no terms that survive the TF-IDF vocabulary
filter. We check whether these movies are concentrated in the cold or
tail groups, where content-based recommendations matter most.

This is a coverage diagnostic, not a model-performance evaluation.

In [13]:
title_has_features = np.asarray(
    T_title.getnnz(axis=1) > 0
).ravel()

title_coverage_df = pd.DataFrame({
    "bucket": pd.Categorical(
        bucket.to_numpy(),
        categories=labels,
        ordered=True,
    ),
    "has_title_features": title_has_features,
})

title_coverage = title_coverage_df.groupby(
    "bucket",
    observed=False,
).agg(
    movies=("has_title_features", "size"),
    movies_with_title_features=("has_title_features", "sum"),
    pct_with_title_features=("has_title_features", "mean"),
)

title_coverage["pct_with_title_features"] *= 100

display(title_coverage.round(2))

,movies,movies_with_title_features,pct_with_title_features
bucket,,,
0,1095,960,87.67
1-4,5666,5002,88.28
5-19,4836,4320,89.33
20-99,3299,2978,90.27
100-499,1685,1528,90.68
500+,660,603,91.36


In [14]:
zero_title_idx = meta.index[~title_has_features]

display(
    meta.loc[
        zero_title_idx,
        ["title", "title_clean", "genres"],
    ].head(20)
)

,title,title_clean,genres
i,,,
9,GoldenEye (1995),GoldenEye,Action|Adventure|Thriller
21,Copycat (1995),Copycat,Crime|Drama|Horror|Mystery|Thriller
34,Carrington (1995),Carrington,Drama|Romance
38,Clueless (1995),Clueless,Comedy|Romance
42,Restoration (1995),Restoration,Drama
47,Pocahontas (1995),Pocahontas,Animation|Children|Drama|Musical|Romance
52,Lamerica (1994),Lamerica,Adventure|Drama
106,Catwalk (1996),Catwalk,Documentary
107,Braveheart (1995),Braveheart,Action|Drama|War


## 4 · Reconstruct the frozen content-v1 features

Content-v1 uses multi-hot genre features and one-hot release-decade
features. The selected configuration uses equal genre and decade
multipliers, disables IDF weighting, and applies L2 normalization.

We reconstruct this exact representation so the title-enhanced
candidates differ only in the addition and weight of title features.

In [15]:
# Recreate the same genre features used in notebook 05.
year_v1 = meta["title"].str.extract(
    r"\((\d{4})\)\s*$"
)[0].astype(float)

genres_v1 = (
    meta["genres"]
    .str.get_dummies("|")
    .drop(columns="(no genres listed)", errors="ignore")
    .astype("int8")
)

decade_v1 = (year_v1 // 10 * 10).clip(lower=1910)

decades_v1 = pd.get_dummies(
    decade_v1.map(
        lambda d: f"{int(d)}s" if pd.notna(d) else None
    )
).astype("int8")

B_content_v1 = pd.concat(
    [genres_v1, decades_v1],
    axis=1,
)

# The frozen v1 configuration has IDF weighting disabled.
X_content_v1 = B_content_v1.to_numpy(dtype=np.float32)

v1_norms = np.linalg.norm(
    X_content_v1,
    axis=1,
    keepdims=True,
)

X_content_v1 = np.divide(
    X_content_v1,
    v1_norms,
    out=np.zeros_like(X_content_v1),
    where=v1_norms > 0,
)

print("Genre features:", genres_v1.shape[1])
print("Decade features:", decades_v1.shape[1])
print("Frozen v1 matrix:", X_content_v1.shape)
print("Movies with no v1 features:", int((v1_norms.ravel() == 0).sum()))

assert X_content_v1.shape == (n_items, 31)
assert np.isfinite(X_content_v1).all()
assert int((v1_norms.ravel() == 0).sum()) == 15

Genre features: 19
Decade features: 12
Frozen v1 matrix: (17241, 31)
Movies with no v1 features: 15


### 4.1 · Fix the title-feature weight grid

We test five predetermined title-feature shares, including the frozen
content-v1 reference and a title-only diagnostic.

For combined models, the genre-and-decade block and the title block
are scaled before the complete vector is normalized. We use square
roots of the shares so the parameter represents the title block's
share of squared vector norm when both blocks are present.

The grid is fixed before evaluation. We will not change it in response
to the observed validation scores.

In [19]:
from scipy import sparse as sp

TITLE_SHARES = [0.0, 0.10, 0.25, 0.50, 1.0]

X_v1_sparse = sp.csr_matrix(
    X_content_v1,
    dtype=np.float32,
)

T_title_sparse = T_title.tocsr().astype(np.float32)


def combine_title_features(title_share):
    """Combine frozen v1 features with title TF-IDF features."""

    if not 0.0 <= title_share <= 1.0:
        raise ValueError("title_share must be between 0 and 1.")

    # Zero title weight means exactly the frozen content-v1 model.
    if title_share == 0.0:
        return X_v1_sparse.copy()

    base_scale = np.float32(np.sqrt(1.0 - title_share))
    title_scale = np.float32(np.sqrt(title_share))

    combined = sp.hstack(
        [
            base_scale * X_v1_sparse,
            title_scale * T_title_sparse,
        ],
        format="csr",
    ).astype(np.float32)

    norms = np.sqrt(
        np.asarray(
            combined.multiply(combined).sum(axis=1)
        ).ravel()
    )

    inverse_norms = np.divide(
        1.0,
        norms,
        out=np.zeros_like(norms, dtype=np.float32),
        where=norms > 0,
    )

    combined = sp.diags(inverse_norms) @ combined

    return combined.tocsr()

### 4.2 · Build the candidate feature matrices

We now build one feature matrix for each fixed title share. We verify
that the zero-share candidate exactly reproduces content-v1 and record
the shape and number of featureless movies for every candidate.

No recommendation metrics are calculated in this step.

In [20]:
title_candidates = {}

for share in TITLE_SHARES:
    X_candidate = combine_title_features(share)
    title_candidates[share] = X_candidate

    row_norms = np.sqrt(
        np.asarray(
            X_candidate.multiply(X_candidate).sum(axis=1)
        ).ravel()
    )

    print(
        f"title share={share:>4.2f} | "
        f"shape={X_candidate.shape} | "
        f"non-zero entries={X_candidate.nnz:,} | "
        f"featureless movies={(row_norms == 0).sum():,}"
    )

# The zero-share candidate must reproduce frozen content-v1.
assert np.allclose(
    title_candidates[0.0].toarray(),
    X_content_v1,
    atol=1e-6,
)

print("Default feature-matrix consistency check passed.")

title share=0.00 | shape=(17241, 31) | non-zero entries=53,959 | featureless movies=15
title share=0.10 | shape=(17241, 9398) | non-zero entries=118,675 | featureless movies=2
title share=0.25 | shape=(17241, 9398) | non-zero entries=118,675 | featureless movies=2
title share=0.50 | shape=(17241, 9398) | non-zero entries=118,675 | featureless movies=2
title share=1.00 | shape=(17241, 9398) | non-zero entries=64,716 | featureless movies=1,850
Default feature-matrix consistency check passed.


## 5 · Build user profiles for the title-enhanced candidates

Each movie now has a feature vector combining genres, decade and,
depending on the candidate, title TF-IDF features.

We build user profiles by averaging the feature vectors of movies
rated 4 or higher in the training data. Every candidate gets its own
profiles, ensuring that the profile representation matches the movie
features used to score recommendations.

Validation and test ratings are not used to construct these profiles.

In [39]:
def build_profiles_for(df, X_features):
    """Average the feature vectors of each user's liked training movies."""

    # Standardize the feature representation to sparse CSR format.
    X_features = sp.csr_matrix(
        X_features,
        dtype=np.float32,
    )

    liked = df.loc[df["rating"] >= REL]

    L = sp.csr_matrix(
        (
            np.ones(len(liked), dtype=np.float32),
            (
                liked["u"].to_numpy(),
                liked["i"].to_numpy(),
            ),
        ),
        shape=(n_users, n_items),
    )

    # Number of positively rated training movies per user.
    n_liked = np.asarray(L.sum(axis=1)).ravel()

    # Average each user's liked-movie features.
    inv = np.divide(
        1.0,
        n_liked,
        out=np.zeros_like(n_liked),
        where=n_liked > 0,
    )

    profiles = (sp.diags(inv) @ L @ X_features).tocsr()

    return profiles, n_liked


title_profiles = {}
profile_counts = {}

for share, X_candidate in title_candidates.items():
    profiles_candidate, n_liked_candidate = build_profiles_for(
        train,
        X_candidate,
    )

    title_profiles[share] = profiles_candidate
    profile_counts[share] = n_liked_candidate

    print(
        f"Title share={share:.2f} | "
        f"profile shape={profiles_candidate.shape} | "
        f"non-zero values={profiles_candidate.nnz:,} | "
        f"users with no liked training movie="
        f"{(n_liked_candidate == 0).sum():,}"
    )

Title share=0.00 | profile shape=(20093, 31) | non-zero values=387,044 | users with no liked training movie=34
Title share=0.10 | profile shape=(20093, 9398) | non-zero values=4,194,835 | users with no liked training movie=34
Title share=0.25 | profile shape=(20093, 9398) | non-zero values=4,194,835 | users with no liked training movie=34
Title share=0.50 | profile shape=(20093, 9398) | non-zero values=4,194,835 | users with no liked training movie=34
Title share=1.00 | profile shape=(20093, 9398) | non-zero values=3,807,791 | users with no liked training movie=34


### 5.1 · Validate the user profiles

Before scoring recommendations, we check that every profile matrix
has the expected user and feature dimensions, contains finite values,
and uses the same positive-training interaction counts.

Because all candidates use the same training interactions, the number
of liked movies per user must remain unchanged across configurations.

In [40]:
# The number of liked training movies must be identical across candidates.
reference_counts = profile_counts[0.0]

for share, counts_for_share in profile_counts.items():
    assert np.array_equal(reference_counts, counts_for_share), (
        f"Positive interaction counts differ at title share {share}."
    )

    profiles_candidate = title_profiles[share]
    X_candidate = title_candidates[share]

    assert profiles_candidate.shape == (
        n_users,
        X_candidate.shape[1],
    )

    assert np.isfinite(profiles_candidate.data).all()

assert np.array_equal(
    title_profiles[0.0].toarray(),
    (
        sp.diags(
            np.divide(
                1.0,
                reference_counts,
                out=np.zeros_like(reference_counts),
                where=reference_counts > 0,
            )
        )
        @ sp.csr_matrix(
            (
                np.ones(
                    len(train.loc[train.rating >= REL]),
                    dtype=np.float32,
                ),
                (
                    train.loc[train.rating >= REL, "u"].to_numpy(),
                    train.loc[train.rating >= REL, "i"].to_numpy(),
                ),
            ),
            shape=(n_users, n_items),
        )
        @ X_v1_sparse
    ).toarray()
)

print("All five profile matrices passed consistency checks.")
print(
    "Users with no liked training movie:",
    int((reference_counts == 0).sum()),
)

All five profile matrices passed consistency checks.
Users with no liked training movie: 34


## 6 · Score movies for each title-feature configuration

We now use each user's profile to score every movie in the catalogue.
The score is the dot product between a user profile and a movie's
feature vector. Higher scores indicate greater feature similarity.

We add a tiny random value to break ties reproducibly, using the same
fixed random seeds for every configuration. Seen movies will be
excluded by the existing ranking helper.

The scorer returns dense score batches, while feature matrices and
user profiles remain sparse to conserve memory.

In [41]:
rng = np.random.default_rng(SEED)


def make_title_scorer(title_share):
    """Return a scorer using the matching profile and item matrices."""

    item_features = title_candidates[title_share]
    user_profiles = title_profiles[title_share]

    def score(user_batch):
        scores = (
            user_profiles[user_batch] @ item_features.T
        ).toarray().astype(np.float32, copy=False)

        # Break ties using a reproducible, very small perturbation.
        scores += 1e-6 * rng.random(
            scores.shape,
            dtype=np.float32,
        )

        return scores

    return score

### 6.1 · Verify the scoring function

Before running the full experiment, we score a few validation users.
The resulting matrix should contain one row per user and one column
per movie, with finite values throughout.

This is a technical check, not a recommendation-quality evaluation.

In [42]:
from recutils import rank_topk, to_sets
from scipy import sparse as sp

K = 10
N_TIE_SEEDS = 10

val_rel_df = val.loc[val["rating"] >= REL]
rel_val = to_sets(val_rel_df)

sample_users = np.asarray(
    list(rel_val.keys())[:3],
    dtype=np.int32,
)

rng = np.random.default_rng(SEED)

sample_scores = make_title_scorer(0.25)(sample_users)

print("Sample score matrix:", sample_scores.shape)
print("Score range:", float(sample_scores.min()),
      float(sample_scores.max()))

assert sample_scores.shape == (3, n_items)
assert np.isfinite(sample_scores).all()

Sample score matrix: (3, 17241)
Score range: 2.102017448990523e-09 0.4520425796508789


## 7 · Set up the fixed validation protocol

We evaluate movies that users rated 4 or higher in validation.
Seen training movies are excluded from recommendations.

Cold movies have fewer than 5 positive training ratings, and tail
movies have fewer than 20. The rarity groups are defined from training
data, not from validation outcomes.

View A ranks all eligible unseen movies. View B restricts the candidates
to the selected rarity slice.

We use the same protocol for every title-feature configuration so that
differences reflect the feature representation rather than a change
in the evaluation setup.

In [43]:
# All movies rated in train are excluded from recommendations.
seen_train = sp.csr_matrix(
    (
        np.ones(len(train), dtype=np.float32),
        (train["u"].to_numpy(), train["i"].to_numpy()),
    ),
    shape=(n_users, n_items),
)

# Use the existing positive-count array calculated from train.
SLICES = {
    "cold": 5,
    "tail": 20,
}

in_slice = {
    ("val", name): n_pos < threshold
    for name, threshold in SLICES.items()
}

def slice_rel(relevant, mask):
    """Keep each user's relevant movies within a rarity slice."""

    filtered = {
        u: {movie for movie in movies if mask[movie]}
        for u, movies in relevant.items()
    }

    return {
        u: movies
        for u, movies in filtered.items()
        if movies
    }


def per_user(ranked, relevant):
    """Calculate NDCG@10 and Recall@10 for each user."""

    discounts = 1 / np.log2(np.arange(2, K + 2))
    recalls, ndcgs = [], []

    for u, relevant_movies in relevant.items():
        top = ranked[u][:K]

        gains = np.array(
            [movie in relevant_movies for movie in top],
            dtype=float,
        )

        recalls.append(gains.sum() / len(relevant_movies))

        ideal_length = min(len(relevant_movies), K)
        ideal_dcg = discounts[:ideal_length].sum()
        actual_dcg = (gains * discounts[:len(top)]).sum()

        ndcgs.append(actual_dcg / ideal_dcg)

    return {
        "Recall@10": np.asarray(recalls),
        "NDCG@10": np.asarray(ndcgs),
    }


print("Validation users with relevant movies:", len(rel_val))
print("Seen-training matrix shape:", seen_train.shape)

for slice_name in SLICES:
    n_slice_users = len(
        slice_rel(rel_val, in_slice[("val", slice_name)])
    )
    print(f"{slice_name} evaluation users: {n_slice_users:,}")

assert seen_train.shape == (n_users, n_items)
assert n_pos.shape[0] == n_items

Validation users with relevant movies: 18392
Seen-training matrix shape: (20093, 17241)
cold evaluation users: 676
tail evaluation users: 2,144


## 8 · Evaluate title-feature configurations

We compare the frozen content-v1 representation with four predetermined
title-feature weights. For each configuration, we evaluate the cold and
tail slices under Views A and B.

Each run uses the same fixed tie-breaking seeds. No test interactions
are used to build profiles, choose configurations or calculate scores.

The primary selection metric is tail-slice NDCG@10 in View B.
Cold performance and View A results are reported alongside it.

In [44]:
def evaluate_title_configuration(title_share, slice_name, view):
    global rng

    mask = in_slice[("val", slice_name)]
    relevant = slice_rel(rel_val, mask)

    if not relevant:
        raise ValueError(
            f"No relevant validation users for {slice_name}."
        )

    scorer = make_title_scorer(title_share)
    runs = []

    for seed in range(N_TIE_SEEDS):
        # The same seeds are reused across configurations.
        rng = np.random.default_rng(seed)

        def score_batch(user_batch):
            scores = scorer(user_batch)

            if view == "B":
                scores[:, ~mask] = -np.inf

            return scores

        ranked = rank_topk(
            score_batch,
            relevant.keys(),
            seen_train,
            K,
            batch=256,
        )

        runs.append(per_user(ranked, relevant))

    averaged = {
        metric: np.mean(
            [run[metric] for run in runs],
            axis=0,
        )
        for metric in runs[0]
    }

    return averaged, len(relevant)

In [45]:
title_evaluations = {}

for title_share in TITLE_SHARES:
    for slice_name in ("tail", "cold"):
        for view in ("A", "B"):

            metrics, n_eval_users = evaluate_title_configuration(
                title_share,
                slice_name,
                view,
            )

            title_evaluations[
                (title_share, slice_name, view)
            ] = {
                "metrics": metrics,
                "n_users": n_eval_users,
            }

            print(
                f"title_share={title_share:.2f} | "
                f"{slice_name} | View {view} | "
                f"NDCG@10="
                f"{metrics['NDCG@10'].mean():.5f} | "
                f"Recall@10="
                f"{metrics['Recall@10'].mean():.5f}"
            )

title_share=0.00 | tail | View A | NDCG@10=0.00347 | Recall@10=0.00671
title_share=0.00 | tail | View B | NDCG@10=0.00587 | Recall@10=0.01162
title_share=0.00 | cold | View A | NDCG@10=0.00173 | Recall@10=0.00449
title_share=0.00 | cold | View B | NDCG@10=0.00565 | Recall@10=0.01114
title_share=0.10 | tail | View A | NDCG@10=0.00191 | Recall@10=0.00337
title_share=0.10 | tail | View B | NDCG@10=0.00427 | Recall@10=0.00899
title_share=0.10 | cold | View A | NDCG@10=0.00216 | Recall@10=0.00288
title_share=0.10 | cold | View B | NDCG@10=0.00483 | Recall@10=0.00814
title_share=0.25 | tail | View A | NDCG@10=0.00135 | Recall@10=0.00300
title_share=0.25 | tail | View B | NDCG@10=0.00302 | Recall@10=0.00695
title_share=0.25 | cold | View A | NDCG@10=0.00221 | Recall@10=0.00296
title_share=0.25 | cold | View B | NDCG@10=0.00556 | Recall@10=0.01095
title_share=0.50 | tail | View A | NDCG@10=0.00088 | Recall@10=0.00154
title_share=0.50 | tail | View B | NDCG@10=0.00187 | Recall@10=0.00409
title_

## 9 · Quantify uncertainty and compare against content-v1

A slightly higher average NDCG@10 does not automatically mean a
configuration is better. We use paired user-level bootstrap intervals
to quantify uncertainty in the difference against the frozen model.

The title feature is retained only if the paired 95% confidence
interval for its tail View B NDCG@10 improvement excludes zero on
the positive side. Otherwise, we record no measurable gain.

In [46]:
N_BOOTSTRAP = 1000
bootstrap_rng = np.random.default_rng(SEED)

bootstrap_indices = {}

for slice_name in ("tail", "cold"):
    n_eval_users = title_evaluations[
        (0.0, slice_name, "B")
    ]["n_users"]

    bootstrap_indices[slice_name] = bootstrap_rng.integers(
        0,
        n_eval_users,
        size=(N_BOOTSTRAP, n_eval_users),
    )


def bootstrap_ci(values, indices):
    bootstrap_means = values[indices].mean(axis=1)

    return (
        float(values.mean()),
        float(np.percentile(bootstrap_means, 2.5)),
        float(np.percentile(bootstrap_means, 97.5)),
    )


rows = []

for title_share in TITLE_SHARES:
    for slice_name in ("tail", "cold"):
        indices = bootstrap_indices[slice_name]

        for view in ("A", "B"):
            current = title_evaluations[
                (title_share, slice_name, view)
            ]["metrics"]

            baseline = title_evaluations[
                (0.0, slice_name, view)
            ]["metrics"]

            for metric in ("NDCG@10", "Recall@10"):
                mean, low, high = bootstrap_ci(
                    current[metric],
                    indices,
                )

                delta = current[metric] - baseline[metric]
                delta_mean, delta_low, delta_high = bootstrap_ci(
                    delta,
                    indices,
                )

                rows.append({
                    "title_share": title_share,
                    "slice": slice_name,
                    "view": view,
                    "n_users": len(current[metric]),
                    "metric": metric,
                    "estimate": mean,
                    "ci95_low": low,
                    "ci95_high": high,
                    "difference_vs_content_v1": delta_mean,
                    "difference_ci95_low": delta_low,
                    "difference_ci95_high": delta_high,
                })

title_results = pd.DataFrame(rows)

display(
    title_results[
        (title_results["slice"] == "tail")
        & (title_results["view"] == "B")
        & (title_results["metric"] == "NDCG@10")
    ]
    .sort_values("estimate", ascending=False)
    .round(6)
)

,title_share,slice,view,n_users,metric,estimate,ci95_low,ci95_high,difference_vs_content_v1,difference_ci95_low,difference_ci95_high
2,0.00,tail,B,2144,NDCG@10,0.005873,0.003935,0.007986,0.000000,0.000000,0.000000
10,0.10,tail,B,2144,NDCG@10,0.004268,0.002548,0.006141,-0.001605,-0.003258,-0.000108
34,1.00,tail,B,2144,NDCG@10,0.003845,0.002078,0.006004,-0.002028,-0.004903,0.000660
18,0.25,tail,B,2144,NDCG@10,0.003023,0.001824,0.004267,-0.002850,-0.004818,-0.000933
26,0.50,tail,B,2144,NDCG@10,0.001869,0.000977,0.002828,-0.004004,-0.006157,-0.001950


### 9.1 · Save the title-feature validation results

We save all five configurations, across both views and both rarity
slices, including their confidence intervals and paired differences.
This preserves the complete experiment for review before any
content-v2 configuration is accepted.

In [47]:
reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

title_report_path = (
    reports_dir / "content_v2a_title_validation.csv"
)

title_results.to_csv(title_report_path, index=False)

print("Saved:", title_report_path)

Saved: ..\reports\content_v2a_title_validation.csv


### The frozen content-v1 baseline outperforms every title-enhanced configuration in this experiment.

## 10 · Designing a temporally controlled tag experiment

External MovieLens tags cover most rare movies, but a tag might have
been created after the rating we are trying to predict.

We will choose a fixed historical cutoff using data-availability
diagnostics before examining model scores. Tag features will use only
tags created on or before that cutoff, and we will evaluate only
eligible validation interactions occurring after it.

The frozen content-v1 model will be evaluated on the exact same
subset, so we can measure the incremental contribution of tags.

This is a controlled temporal subset experiment, not a fully global
historical replay of every user's training history.

In [48]:
# Inspect the yearly distribution of external tags.
tag_yearly = (
    pd.to_datetime(
        external_tag_events["timestamp"],
        unit="s",
    )
    .dt.year
    .value_counts()
    .sort_index()
    .rename("external_tag_records")
    .to_frame()
)

# Count positive tail validation interactions by year.
tail_mask = n_pos < 20

tail_val = val.loc[
    (val["rating"] >= REL)
    & val["i"].isin(np.flatnonzero(tail_mask))
].copy()

tail_val["year"] = (
    pd.to_datetime(tail_val["timestamp"], unit="s")
    .dt.year
)

tail_val_yearly = (
    tail_val.groupby("year")
    .agg(
        tail_validation_interactions=("i", "size"),
        eligible_users=("u", "nunique"),
    )
)

yearly_diagnostics = tag_yearly.join(
    tail_val_yearly,
    how="outer",
).fillna(0).astype(int)

display(yearly_diagnostics)

,external_tag_records,tail_validation_interactions,eligible_users
timestamp,,,
1996,0,157,112
1997,0,99,72
1998,0,29,25
1999,0,67,52
2000,0,129,99
2001,0,96,78
2002,0,90,62
2003,0,78,60
2004,0,69,53


### 10.1 · Fix the temporal cutoff

We choose 31 December 2015 as the cutoff based on tag availability
and the number of later tail validation interactions, before examining
any tag-model scores.

Tag features will use only external tags created before 1 January 2016.
Evaluation will include only relevant tail validation interactions
dated on or after 1 January 2016.

We will evaluate frozen content-v1 on exactly the same eligible
users and targets. This controls the comparison for the evaluation
subset, although the existing per-user split is not a full global
historical replay.

In [49]:
# 10.2 · Apply the fixed temporal cutoff

CUTOFF_DATE = pd.Timestamp("2016-01-01", tz="UTC")
CUTOFF_TS = int(CUTOFF_DATE.timestamp())

# Use external tags created strictly before 2016.
tag_source = external_tag_events.loc[
    external_tag_events["timestamp"] < CUTOFF_TS
].copy()

tag_source["tag"] = (
    tag_source["tag"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

tag_source = tag_source.loc[
    tag_source["tag"].ne("")
]

# Relevant validation interactions in the fixed tail slice.
tail_mask = n_pos < 20

tail_val_all = val.loc[
    (val["rating"] >= REL)
    & val["i"].isin(np.flatnonzero(tail_mask))
].copy()

# Evaluate only targets dated on or after the cutoff.
tail_val_temporal = tail_val_all.loc[
    tail_val_all["timestamp"] >= CUTOFF_TS
].copy()

print("Cutoff date:", CUTOFF_DATE.date())
print("Eligible external tag records:", f"{len(tag_source):,}")
print("Movies with at least one eligible tag:",
      f"{tag_source['i'].nunique():,}")
print("Tail validation interactions before cutoff:",
      f"{(tail_val_all['timestamp'] < CUTOFF_TS).sum():,}")
print("Eligible tail validation interactions:",
      f"{len(tail_val_temporal):,}")
print("Unique eligible validation users:",
      f"{tail_val_temporal['u'].nunique():,}")
print("Unique eligible target movies:",
      f"{tail_val_temporal['i'].nunique():,}")

# Verify temporal eligibility.
assert tag_source["timestamp"].max() < CUTOFF_TS
assert tail_val_temporal["timestamp"].min() >= CUTOFF_TS

# Verify the tag source excludes sampled users.
assert not tag_source["userId"].isin(sampled_user_ids).any()

Cutoff date: 2016-01-01
Eligible external tag records: 385,780
Movies with at least one eligible tag: 11,850
Tail validation interactions before cutoff: 1,634
Eligible tail validation interactions: 1,585
Unique eligible validation users: 948
Unique eligible target movies: 1,266


## 11 · Build temporally controlled tag features

We use only external MovieLens tags created before 1 January 2016.
Tags from users in our sampled population are excluded.

Each movie becomes a document containing its distinct tag labels.
We preserve multiword tags such as "twist ending" as single features
rather than splitting them into individual words.

TF-IDF gives greater weight to tags that distinguish movies from one
another. We then inspect feature coverage and verify that every feature
row corresponds to the correct movie in our catalogue.

This creates a candidate feature representation; it does not yet
establish that tags improve recommendation quality.

In [50]:
# 11.1 · Create one copy of each tag per movie

tag_pairs = tag_source[["i", "tag"]].copy()

tag_pairs["tag"] = (
    tag_pairs["tag"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.lower()
)

tag_pairs = tag_pairs.loc[tag_pairs["tag"].ne("")]

tag_pairs = tag_pairs.drop_duplicates(
    subset=["i", "tag"]
)

print("Original eligible tag records:", f"{len(tag_source):,}")
print("Distinct movie-tag pairs:", f"{len(tag_pairs):,}")
print("Movies represented:", f"{tag_pairs['i'].nunique():,}")

assert not tag_pairs.duplicated(["i", "tag"]).any()
assert not tag_source["userId"].isin(sampled_user_ids).any()
assert tag_source["timestamp"].max() < CUTOFF_TS

Original eligible tag records: 385,780
Distinct movie-tag pairs: 154,747
Movies represented: 11,850


### 11.1 · Assemble the movie tag documents

We combine the distinct tags of each movie into one document.
Movies without eligible tags receive an empty document so the
catalogue remains aligned with the existing item indices.

A separator preserves each complete tag as one feature, including
multiword phrases.

In [51]:
# Group distinct tag labels by internal movie index.
tag_docs_by_movie = (
    tag_pairs.groupby("i", sort=False)["tag"]
    .agg("\x1f".join)
)

# Keep every movie in index order, including movies without tags.
tag_docs = (
    tag_docs_by_movie
    .reindex(np.arange(n_items), fill_value="")
    .fillna("")
    .tolist()
)

print("Tag documents:", len(tag_docs))
print("Non-empty documents:", sum(bool(doc) for doc in tag_docs))
print("Empty documents:", sum(not doc for doc in tag_docs))

assert len(tag_docs) == n_items
assert meta.index.to_numpy().tolist() == list(range(n_items))

Tag documents: 17241
Non-empty documents: 11850
Empty documents: 5391


### 11.2 · Convert tags into TF-IDF features

Each distinct tag becomes a possible feature. We discard tags that
occur in only one movie because they cannot create similarity between
different movies in this representation.

Inverse document frequency gives less weight to tags used across
many movies and more weight to tags that distinguish a smaller group.
Each non-empty movie vector is normalized to unit length.

In [52]:
from sklearn.feature_extraction.text import TfidfVectorizer

def split_tag_tokens(document):
    if not document:
        return []
    return document.split("\x1f")


tag_vectorizer = TfidfVectorizer(
    tokenizer=split_tag_tokens,
    token_pattern=None,
    preprocessor=None,
    lowercase=False,
    ngram_range=(1, 1),
    min_df=2,
    norm="l2",
    dtype=np.float32,
)

T_tags = tag_vectorizer.fit_transform(tag_docs)

print("Tag matrix shape:", T_tags.shape)
print("Tag vocabulary size:", len(tag_vectorizer.vocabulary_))
print("Non-zero entries:", f"{T_tags.nnz:,}")
print(
    "Matrix density:",
    f"{T_tags.nnz / (T_tags.shape[0] * T_tags.shape[1]):.4%}",
)

tag_has_features = np.asarray(
    T_tags.getnnz(axis=1) > 0
).ravel()

tag_norms = np.sqrt(
    np.asarray(T_tags.multiply(T_tags).sum(axis=1)).ravel()
)

print("Movies with tag features:", int(tag_has_features.sum()))
print("Movies without tag features:", int((~tag_has_features).sum()))

assert T_tags.shape[0] == n_items
assert T_tags.shape[1] > 0
assert np.isfinite(T_tags.data).all()
assert np.allclose(tag_norms[tag_has_features], 1.0, atol=1e-5)

Tag matrix shape: (17241, 10715)
Tag vocabulary size: 10715
Non-zero entries: 137,995
Matrix density: 0.0747%
Movies with tag features: 11764
Movies without tag features: 5477


In [53]:
tag_feature_coverage = pd.DataFrame({
    "bucket": pd.Categorical(
        bucket.to_numpy(),
        categories=labels,
        ordered=True,
    ),
    "has_tag_features": tag_has_features,
})

tag_coverage_after_vectorizing = (
    tag_feature_coverage
    .groupby("bucket", observed=False)
    .agg(
        movies=("has_tag_features", "size"),
        movies_with_features=("has_tag_features", "sum"),
        pct_with_features=("has_tag_features", "mean"),
    )
)

tag_coverage_after_vectorizing["pct_with_features"] *= 100

display(tag_coverage_after_vectorizing.round(2))

,movies,movies_with_features,pct_with_features
bucket,,,
0,1095,460,42.01
1-4,5666,2801,49.44
5-19,4836,3419,70.70
20-99,3299,2862,86.75
100-499,1685,1574,93.41
500+,660,648,98.18


### 11.3 · Combine tag features with content-v1

We combine the frozen genre-and-decade features with the new
temporally controlled tag TF-IDF features.

We test five predetermined tag weights, including the exact
content-v1 baseline. The combined vectors are normalized so that
the score scale remains comparable across movies.

Movies without usable tags retain their original genre and decade
features. We will judge the candidate models using validation results,
not feature coverage alone.

In [54]:
# 11.3 · Build the combined feature matrices

TAG_SHARES = [0.0, 0.10, 0.25, 0.50, 0.75]

X_v1_sparse = sp.csr_matrix(
    X_content_v1,
    dtype=np.float32,
)

T_tags_sparse = T_tags.tocsr().astype(np.float32)

tag_candidates = {}


def combine_tag_features(tag_share):
    """Combine frozen content-v1 and temporally controlled tag features."""

    if not 0.0 <= tag_share <= 1.0:
        raise ValueError("tag_share must be between 0 and 1.")

    # Zero tag weight must exactly preserve the frozen baseline.
    if tag_share == 0.0:
        return X_v1_sparse.copy()

    base_scale = np.float32(np.sqrt(1.0 - tag_share))
    tag_scale = np.float32(np.sqrt(tag_share))

    combined = sp.hstack(
        [
            base_scale * X_v1_sparse,
            tag_scale * T_tags_sparse,
        ],
        format="csr",
    ).astype(np.float32)

    # Normalize each combined movie vector.
    norms = np.sqrt(
        np.asarray(
            combined.multiply(combined).sum(axis=1)
        ).ravel()
    )

    inverse_norms = np.divide(
        1.0,
        norms,
        out=np.zeros_like(norms, dtype=np.float32),
        where=norms > 0,
    )

    combined = sp.diags(inverse_norms) @ combined

    return combined.tocsr()


for share in TAG_SHARES:
    X_candidate = combine_tag_features(share)
    tag_candidates[share] = X_candidate

    row_norms = np.sqrt(
        np.asarray(
            X_candidate.multiply(X_candidate).sum(axis=1)
        ).ravel()
    )

    print(
        f"tag_share={share:.2f} | "
        f"shape={X_candidate.shape} | "
        f"non-zero entries={X_candidate.nnz:,} | "
        f"featureless movies={(row_norms == 0).sum():,}"
    )


assert np.allclose(
    tag_candidates[0.0].toarray(),
    X_content_v1,
    atol=1e-6,
)

print("Frozen content-v1 consistency check passed.")

tag_share=0.00 | shape=(17241, 31) | non-zero entries=53,959 | featureless movies=15


tag_share=0.10 | shape=(17241, 10746) | non-zero entries=191,954 | featureless movies=15
tag_share=0.25 | shape=(17241, 10746) | non-zero entries=191,954 | featureless movies=15
tag_share=0.50 | shape=(17241, 10746) | non-zero entries=191,954 | featureless movies=15
tag_share=0.75 | shape=(17241, 10746) | non-zero entries=191,954 | featureless movies=15
Frozen content-v1 consistency check passed.


## 12 · Build user profiles for the tag-enhanced models

We now build a separate user profile for each tag-weight configuration.

Each profile is the average of the genre, decade and tag features
belonging to movies that the user rated 4 or higher in the training
data. Validation and test ratings are not used to construct profiles.

The zero-tag-share configuration must reproduce the frozen content-v1
profiles. This ensures that our comparison changes only the feature
representation, not the training interactions.

In [55]:
# 12.1 · Build profiles for each tag configuration

tag_profiles = {}
tag_profile_counts = {}

for share, X_candidate in tag_candidates.items():
    profiles_candidate, n_liked_candidate = build_profiles_for(
        train,
        X_candidate,
    )

    tag_profiles[share] = profiles_candidate
    tag_profile_counts[share] = n_liked_candidate

    print(
        f"tag_share={share:.2f} | "
        f"profile shape={profiles_candidate.shape} | "
        f"non-zero values={profiles_candidate.nnz:,} | "
        f"users with no liked training movie="
        f"{(n_liked_candidate == 0).sum():,}"
    )

tag_share=0.00 | profile shape=(20093, 31) | non-zero values=387,044 | users with no liked training movie=34
tag_share=0.10 | profile shape=(20093, 10746) | non-zero values=31,658,323 | users with no liked training movie=34
tag_share=0.25 | profile shape=(20093, 10746) | non-zero values=31,658,323 | users with no liked training movie=34
tag_share=0.50 | profile shape=(20093, 10746) | non-zero values=31,658,323 | users with no liked training movie=34
tag_share=0.75 | profile shape=(20093, 10746) | non-zero values=31,658,323 | users with no liked training movie=34


### 12.1 · Validate the user profiles

All configurations use the same training interactions, so their
positive-interaction counts must be identical.

We also verify that profile dimensions match their corresponding
feature matrices and that the zero-tag-share profiles reproduce
the frozen content-v1 profiles.

In [56]:
# Reconstruct the reference profiles from the frozen feature matrix.
profiles_v1_check, n_liked_v1_check = build_profiles_for(
    train,
    X_content_v1,
)

# All configurations must use the same training interaction counts.
for share, counts in tag_profile_counts.items():
    assert np.array_equal(counts, n_liked_v1_check), (
        f"Training interaction counts differ at tag_share={share}."
    )

    profiles_candidate = tag_profiles[share]

    assert profiles_candidate.shape == (
        n_users,
        tag_candidates[share].shape[1],
    )

    assert np.isfinite(profiles_candidate.data).all()

# Compare the zero-tag-share profiles to the frozen baseline.
profile_difference = (
    tag_profiles[0.0] - profiles_v1_check
).tocsr()

if profile_difference.nnz:
    max_difference = np.max(np.abs(profile_difference.data))
else:
    max_difference = 0.0

print(
    "Maximum profile difference from content-v1:",
    float(max_difference),
)

assert max_difference < 1e-6

print("All tag-profile consistency checks passed.")

Maximum profile difference from content-v1: 0.0
All tag-profile consistency checks passed.


Setting up the evaluation data

In [63]:
# The ranking utility expects a dictionary:
# user index -> set of movies already seen in training.
seen_train = sp.csr_matrix(
    (
        np.ones(len(train), dtype=np.float32),
        (train["u"].to_numpy(), train["i"].to_numpy()),
    ),
    shape=(n_users, n_items),
)

# Ensure each stored user-movie interaction is marked as seen.
seen_train.data[:] = 1.0

# Relevant tail-movie validation targets after the fixed cutoff.
rel_tail_temporal = to_sets(tail_val_temporal)

assert len(tail_val_temporal) == 1585
assert len(rel_tail_temporal) == 948

print("Eligible validation interactions:", len(tail_val_temporal))
print("Eligible validation users:", len(rel_tail_temporal))
print(
    "Training users with history:",
    int((np.diff(seen_train.indptr) > 0).sum())
)

Eligible validation interactions: 1585
Eligible validation users: 948
Training users with history: 20093


Create the scoring function
This scores movies in batches, preventing us from allocating one enormous user-by-movie score matrix.

In [64]:
def make_tag_scorer(tag_share, view="A", seed=SEED):
    item_features = tag_candidates[tag_share].tocsr()
    user_profiles = tag_profiles[tag_share].tocsr()
    rng = np.random.default_rng(seed)

    def score(user_batch):
        user_batch = np.asarray(user_batch, dtype=np.int64)

        scores = (
            user_profiles[user_batch] @ item_features.T
        ).toarray().astype(np.float32, copy=False)

        # Tiny, reproducible noise breaks exact score ties.
        scores += 1e-6 * rng.random(
            scores.shape, dtype=np.float32
        )

        if view == "B":
            # Only rare/tail movies can compete.
            scores[:, ~tail_mask] = -np.inf
        elif view != "A":
            raise ValueError("view must be 'A' or 'B'")

        return scores

    return score

We need per-user scores so we can calculate paired confidence intervals later, rather than relying only on aggregate averages.

In [65]:
def per_user_metrics(ranked, relevant, k=K):
    discounts = 1.0 / np.log2(np.arange(2, k + 2))
    rows = {}

    for u, relevant_items in relevant.items():
        top_items = ranked[u][:k]

        hits = np.array(
            [int(i in relevant_items) for i in top_items],
            dtype=np.float32,
        )

        ideal_len = min(len(relevant_items), k)
        idcg = discounts[:ideal_len].sum()
        dcg = (hits * discounts[:len(top_items)]).sum()

        rows[int(u)] = {
            f"NDCG@{k}": dcg / idcg if idcg else 0.0,
            f"Recall@{k}": hits.sum() / len(relevant_items),
        }

    return pd.DataFrame.from_dict(rows, orient="index").sort_index()

Before testing tag enrichment, score the frozen baseline on this exact temporal subset.

In [66]:
ranked_v1_tail_B = rank_topk(
    make_tag_scorer(0.0, view="B"),
    rel_tail_temporal.keys(),
    seen_train,
    K,
    batch=128,
)

metrics_v1_tail_B = per_user_metrics(
    ranked_v1_tail_B,
    rel_tail_temporal,
)

print("Content-v1 — temporal tail validation, View B")
print("Evaluated users:", len(metrics_v1_tail_B))
print(metrics_v1_tail_B.mean().round(6))

Content-v1 — temporal tail validation, View B
Evaluated users: 948
NDCG@10      0.006163
Recall@10    0.011199
dtype: float64


### 13 · Evaluate tag enrichment

Compare the frozen content-v1 model with four tag-enhanced configurations. Each configuration is evaluated on the same temporally eligible tail-movie validation targets.

We repeat ranking with 10 fixed tie-breaking seeds and average the per-user metrics. The primary selection metric is NDCG@10; Recall@10 is reported alongside it.

The zero-tag-share configuration is our control. We will select a tag configuration only if the results justify the additional complexity.

In [67]:
# Step 17.1 — Evaluate all tag weights on temporal tail View B

tag_evaluations = {}

users_eval = list(rel_tail_temporal.keys())

for share in TAG_SHARES:
    print(f"Evaluating tag_share={share:.2f} ...")

    runs = []

    for seed in range(N_TIE_SEEDS):
        ranked = rank_topk(
            make_tag_scorer(
                share,
                view="B",
                seed=seed,
            ),
            users_eval,
            seen_train,
            K,
            batch=128,
        )

        runs.append(
            per_user_metrics(
                ranked,
                rel_tail_temporal,
            )
        )

    # Average each user's metrics across the 10 runs.
    metrics = (
        pd.concat(runs)
        .groupby(level=0)
        .mean()
        .sort_index()
    )

    assert len(metrics) == 948
    assert np.isfinite(metrics.to_numpy()).all()

    tag_evaluations[share] = metrics

    print(
        f"  NDCG@10: {metrics['NDCG@10'].mean():.6f} | "
        f"Recall@10: {metrics['Recall@10'].mean():.6f}"
    )

print("\nAll tag configurations evaluated.")

Evaluating tag_share=0.00 ...
  NDCG@10: 0.004986 | Recall@10: 0.009608
Evaluating tag_share=0.10 ...
  NDCG@10: 0.003295 | Recall@10: 0.006224
Evaluating tag_share=0.25 ...
  NDCG@10: 0.002521 | Recall@10: 0.005204
Evaluating tag_share=0.50 ...
  NDCG@10: 0.002594 | Recall@10: 0.004931
Evaluating tag_share=0.75 ...
  NDCG@10: 0.002774 | Recall@10: 0.005846

All tag configurations evaluated.


The tag-enriched models performed worse.

In [68]:
# Compare every tag configuration against content-v1.
# Resample users, preserving the pairing between models.

BASE_SHARE = 0.0
N_BOOT = 10_000

baseline_metrics = tag_evaluations[BASE_SHARE].sort_index()
rng = np.random.default_rng(2026)

# The same sampled users are used for every model comparison.
boot_indices = rng.integers(
    0,
    len(baseline_metrics),
    size=(N_BOOT, len(baseline_metrics)),
)

rows = []

for share in TAG_SHARES:
    if share == BASE_SHARE:
        continue

    candidate_metrics = tag_evaluations[share].sort_index()

    assert candidate_metrics.index.equals(baseline_metrics.index)

    for metric in ["NDCG@10", "Recall@10"]:
        differences = (
            candidate_metrics[metric]
            - baseline_metrics[metric]
        ).to_numpy(dtype=float)

        bootstrap_means = differences[boot_indices].mean(axis=1)
        ci_low, ci_high = np.quantile(
            bootstrap_means, [0.025, 0.975]
        )

        rows.append({
            "tag_share": share,
            "metric": metric,
            "mean_difference": differences.mean(),
            "CI_95_low": ci_low,
            "CI_95_high": ci_high,
        })

tag_ci_results = pd.DataFrame(rows)

display(
    tag_ci_results.round(6)
)

,tag_share,metric,mean_difference,CI_95_low,CI_95_high
0,0.10,NDCG@10,-0.001691,-0.003857,0.000115
1,0.10,Recall@10,-0.003384,-0.007261,-0.000308
2,0.25,NDCG@10,-0.002465,-0.005505,-0.000131
3,0.25,Recall@10,-0.004404,-0.008931,-0.000607
4,0.50,NDCG@10,-0.002392,-0.005550,0.000150
5,0.50,Recall@10,-0.004677,-0.009344,-0.000686
6,0.75,NDCG@10,-0.002212,-0.005438,0.000409
7,0.75,Recall@10,-0.003762,-0.008826,0.000897


These are unadjusted 95% intervals. Because we tested multiple configurations, treat the apparent drop at 0.25 as evidence rather than definitive proof. No tag configuration has beaten the control on mean NDCG@10 or Recall@10.


Hence, we keep content-v1 frozen.

### Next step is to investigate whether Wikidata can provide useful metadata for rare movies